In [1]:
from src.newkan import NewKAN

model = NewKAN(layers=[2, 10, 1], grid_size=10)
print(model)

NewKAN(
  (layers): ModuleList(
    (0-1): 2 x KANLayer(
      (silu): SiLU()
    )
  )
)


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torch.optim as optim

def train_kan(model, X_train, y_train, X_test, y_test, epochs=100, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        
        with torch.no_grad():
            model.eval()
            train_pred = torch.argmax(model(X_train), 1)
            train_acc = (train_pred == y_train).float().mean()
            train_accs.append(train_acc.item())
            
            test_outputs = model(X_test)
            test_loss = criterion(test_outputs, y_test)
            test_losses.append(test_loss.item())
            
            test_pred = torch.argmax(test_outputs, 1)
            test_acc = (test_pred == y_test).float().mean()
            test_accs.append(test_acc.item())
        
        if epoch % 20 == 0:
            print(f'Epoch {epoch}: Train Loss: {loss.item():.4f}, Test Acc: {test_acc.item():.4f}')
    
    return train_losses, test_losses, train_accs, test_accs

In [ ]:
from matplotlib import pyplot as plt

def plot_results(train_loss, test_loss, train_acc, test_acc, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(train_loss, label='Train Loss')
    ax1.plot(test_loss, label='Test Loss')
    ax1.set_title(f'{title} - Loss')
    ax1.legend()
    
    ax2.plot(train_acc, label='Train Accuracy')
    ax2.plot(test_acc, label='Test Accuracy')
    ax2.set_title(f'{title} - Accuracy')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

In [16]:
def print_predictions(model, X, y, dataset_name, num_samples=10):
    print(f"\n=== {dataset_name} Predictions ===")
    model.eval()
    with torch.no_grad():
        predictions = model(X)
        pred_classes = torch.argmax(predictions, 1)
        
        indices = np.random.choice(len(X), min(num_samples, len(X)), replace=False)
        
        print(f"{'Sample':<8} {'Predicted':<12} {'Expected':<12} {'Match':<8}")
        print("-" * 45)
        
        matches = 0
        for idx in indices:
            pred = pred_classes[idx].item()
            actual = y[idx].item()
            match = "✓" if pred == actual else "✗"
            if match == "✓":
                matches += 1
            print(f"{idx:<8} {pred:<12} {actual:<12} {match:<8}")
        
        accuracy = matches / len(indices) * 100
        print("-" * 45)
        print(f"Accuracy on {len(indices)} samples: {accuracy:.1f}%")

In [17]:
model = NewKAN([2, 4, 2])
print(f"KAN parameters: {model.count_params()}")

x_test = torch.randn(4, 2)
print(f"Input shape: {x_test.shape}")
output = model(x_test)
print(f"Output shape: {output.shape}")
print(f"Output values:\n{output.detach()}")

KAN parameters: 172
Input shape: torch.Size([4, 2])
Output shape: torch.Size([4, 2])
Output values:
tensor([[ 4.8743,  5.0807],
        [-0.1289, -0.2645],
        [-0.8173, -0.9686],
        [ 0.1887,  0.0600]])


In [19]:
print("=== XOR Dataset ===")
X_xor = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y_xor = torch.tensor([0,1,1,0], dtype=torch.long)

kan_xor = NewKAN([2, 4, 2])
train_kan(kan_xor, X_xor, y_xor, X_xor, y_xor, epochs=40, lr=0.1)
print_predictions(kan_xor, X_xor, y_xor, "XOR", num_samples=4)


=== XOR Dataset ===
Epoch 0: Train Loss: 0.7252, Test Acc: 0.7500
Epoch 20: Train Loss: 0.0000, Test Acc: 1.0000

=== XOR Predictions ===
Sample   Predicted    Expected     Match   
---------------------------------------------
1        1            1            ✓       
0        0            0            ✓       
2        1            1            ✓       
3        0            0            ✓       
---------------------------------------------
Accuracy on 4 samples: 100.0%


In [ ]:
from sklearn.datasets import load_iris
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split

print("\n=== Iris Dataset ===")
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target

X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42)

X_iris_train = torch.tensor(X_iris_train, dtype=torch.float32)
X_iris_test = torch.tensor(X_iris_test, dtype=torch.float32)
y_iris_train = torch.tensor(y_iris_train, dtype=torch.long)
y_iris_test = torch.tensor(y_iris_test, dtype=torch.long)

kan_iris = NewKAN([4, 8, 3])
print(kan_iris)

train_kan(kan_iris, X_iris_train, y_iris_train, X_iris_test, y_iris_test, epochs=100)
print_predictions(kan_iris, X_iris_test, y_iris_test, "Iris Test Set", num_samples=10)


=== Iris Dataset ===
NewKAN(
  (layers): ModuleList(
    (0-1): 2 x KANLayer(
      (silu): SiLU()
    )
  )
)
Epoch 0: Train Loss: 1.0857, Test Acc: 0.6333
Epoch 20: Train Loss: 0.3555, Test Acc: 0.9667
Epoch 40: Train Loss: 0.0839, Test Acc: 1.0000
Epoch 60: Train Loss: 0.0443, Test Acc: 1.0000
Epoch 80: Train Loss: 0.0282, Test Acc: 1.0000

=== Iris Test Set Predictions ===
Sample   Predicted    Expected     Match   
---------------------------------------------
0        1            1            ✓       
24       2            2            ✓       
16       2            2            ✓       
2        2            2            ✓       
18       1            1            ✓       
3        1            1            ✓       
10       2            2            ✓       
13       0            0            ✓       
4        1            1            ✓       
7        2            2            ✓       
19       2            2            ✓       
26       2            2            ✓       
1